# nb-04 · Publish to CSV  *(stage: PUBLISH)*

**Purpose.** The tidy CSV is the deliverable. Finalize it: full processed audit,
the auto-generated column profile (the data dictionary's sanity check), and the
reconciliation scaffold against each agency's published totals.

| | |
|---|---|
| **Inputs** | `data/processed/reservoir-storage.csv` (from nb-03) |
| **Outputs** | `data/audit/summary-<ts>.md` · `docs/variables.csv` · `data/audit/reconcile.json` |
| **Preconditions** | nb-03 has run. |

A queryable **Datasette** surface + a **Quarto** docs site are deferred (Hub
roadmap) — this notebook stops at the CSV deliverable, as scoped in #9.

In [1]:
# Make the `reservoir` package importable when running from notebooks/.
# Cleaner: `uv pip install -e ..` once, then this cell is a no-op.
import sys, pathlib
SRC = pathlib.Path.cwd().parent / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('package path:', SRC)

package path: /Users/briankeegan/Documents/GitHub/co-environmental-data/pipelines/reservoir-storage/src


In [2]:
from reservoir import audit, config
from IPython.display import Markdown
Markdown(audit.audit_processed())

# Processed audit — 20260618T192325Z

Rows: **3**

## Rows per source × variable

| source | variable | rows | reservoirs | null values |
|---|---|--:|--:|--:|
| dwr_cdss | storage_af | 3 | 1 | 0 |

## Date coverage per source

| source | earliest | latest |
|---|---|---|
| dwr_cdss | 2026-06-15 | 2026-06-17 |

## Value ranges per variable (sanity / outliers)

| variable | unit | min | median | max |
|---|---|--:|--:|--:|
| storage_af | acre-ft | 138,010.0 | 138,120.0 | 138,214.0 |


### Auto-generated column profile → `docs/variables.csv`
If this disagrees with `docs/data-dictionary.md`, the parser is usually wrong.

In [3]:
audit.variables_report()

,column,dtype,distinct,null_rate,sample
0,source,str,1,0.0,dwr_cdss
1,vintage,str,1,0.0,current
2,reservoir_id,str,1,0.0,GREEN MOUNTAIN
3,reservoir_name,str,1,0.0,Green Mountain Reservoir
4,datetime,str,3,0.0,2026-06-15; 2026-06-16; 2026-06-17
5,variable,str,1,0.0,storage_af
6,value,float64,3,0.0,138010.0; 138120.0; 138214.0
7,unit,str,1,0.0,acre-ft
8,qa_flag,str,1,0.0,A
9,concept,str,1,0.0,reservoir.storage_af


### Reconciliation (accuracy vs. the source's own published totals)

Fill `expected` with current storage figures read off each agency's
current-conditions page (see `docs/survey-notes.md`). Any `match == False` beyond
tolerance is a regression to investigate before publishing.

In [4]:
expected = {
    # ('dwr_cdss', 'GREEN MOUNTAIN'): 138214.0,   # ← confirm from the DWR station page
}
audit.reconcile(expected) if expected else print('reconcile: fill expected totals (survey-notes)')

reconcile: fill expected totals (survey-notes)


### The deliverable

In [5]:
import pandas as pd
csv = config.CANONICAL_CSV
df = pd.read_csv(csv)
print('deliverable:', csv.relative_to(config.PROJECT_DIR))
print('rows:', len(df), '| columns:', list(df.columns))
print('load:  pd.read_csv(\'%s\', parse_dates=[\'datetime\'])' % csv.name)
print('wide views → docs/filter-pivot-recipes.md')

deliverable: data/processed/reservoir-storage.csv
rows: 3 | columns: ['source', 'vintage', 'reservoir_id', 'reservoir_name', 'datetime', 'variable', 'value', 'unit', 'qa_flag', 'concept']
load:  pd.read_csv('reservoir-storage.csv', parse_dates=['datetime'])
wide views → docs/filter-pivot-recipes.md
